In [1]:
import torch
def forward_diffusion(x0, t, alpha_bar):
    noise = torch.randn_like(x0)
    sqrt_alpha_bar = torch.sqrt(alpha_bar[t])
    sqrt_one_minus = torch.sqrt(1- alpha_bar[t])

    xt = sqrt_alpha_bar*x0 +sqrt_one_minus*noise
    return xt, noise

In [2]:
import  torch.nn as nn 
class SimpleDenoisor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,64,3,padding =1),
            nn.ReLU(),
            nn.Conv2d(64,64,3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64,3,3,padding=1)
        )
    def forward(self, x,t):
        return self.net(x)

In [4]:
def loss_fn(model, x0, t, alpha_bar):
    xt,noise = forward_diffusion(x0, t, alpha_bar)
    pred_noise = model(xt,t)
    loss = ((noise - pred_noise)**2).mean()
    return loss

In [5]:
model = SimpleDenoisor()
optimizer = torch.optim.Adam(model.parameters(), lr =1e-4)

for epoch in range(epochs):
    for x0 in dataloader:
        t = torch.randint(0,T,(x0.size(0),))
        loss = loss_fn(model, x0,t, alpha_bar)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        


NameError: name 'epochs' is not defined

In [6]:
def sampling(model, shape, alpha, alpha_bar, beta):
    x =torch.randn(shape)
    for t in reversed(range(T)):
        pred_noise = model(x,t)
        alpha_t  = alpha[t]
        alpha_bar_t = alpha_bar[t]
        x = (1 /torch.sqrt(alpha_t))* (
            x - (1-alpha_t)/torch.sqrt(1-alpha_bar_t)*pred_noise
        )
        if t>0:
            x+=torch.sqrt(beta[t])*torch.randn_like(x)
    return x